In [10]:
import pandas as pd

# Load CSVs
audio_df = pd.read_csv("microphone_audio_sensor.csv")
thermal_df = pd.read_csv("thermal_sensor_data.csv")
em_df = pd.read_csv("em_sensor_data.csv")
gps_df = pd.read_csv("gps_environment_data.csv")
proximity_df = pd.read_csv("proximity_sensor_data.csv")
imu_df = pd.read_csv("imu_sensor_data.csv")

# Drop non-feature columns
audio_df.drop(columns=["description", "start_time", "end_time", "audio_pattern"], inplace=True)
thermal_df.drop(columns=["description", "night_vision"], inplace=True)
gps_df.drop(columns=["description", "timestamp", "location_type"], inplace=True)
proximity_df.drop(columns=["description", "timestamp"], inplace=True)
imu_df.drop(columns=["description", "timestamp"], inplace=True)

# Encode categorical values in GPS
gps_df["time_of_day"] = gps_df["time_of_day"].map({"day": 0, "night": 1})
gps_df["movement_status"] = gps_df["movement_status"].map({"moving": 0, "standing": 1, "stationary": 2})

# Flatten thermal matrix
thermal_df["thermal_matrix"] = thermal_df["thermal_matrix"].apply(lambda x: list(map(float, str(x).split(','))))
thermal_expanded = thermal_df["thermal_matrix"].apply(pd.Series)
thermal_expanded.columns = [f"thermal_{i}" for i in thermal_expanded.columns]
thermal_df = pd.concat([thermal_expanded, thermal_df["label"]], axis=1)

# Merge all data
dfs = [audio_df, thermal_df, em_df, gps_df, proximity_df, imu_df]
merged_df = pd.concat(dfs, axis=1)

# Fix label
merged_df["label"] = merged_df.filter(like="label").bfill(axis=1).iloc[:, 0]
merged_df = merged_df.drop(columns=merged_df.columns[merged_df.columns.str.contains("label")][:-1])

# Save or preview
merged_df.to_csv("final_merged_sensor_data.csv", index=False)
merged_df.head()


,peak_db,thermal_0,thermal_1,thermal_2,thermal_3,thermal_4,thermal_5,thermal_6,thermal_7,thermal_8,...,front,left,right,rear,accel_x,accel_y,accel_z,gyro_x,gyro_y,gyro_z
0,41.0,22.0,22.0,23.0,22.0,22.0,22.0,34.0,36.0,34.0,...,1.8,NaN,NaN,NaN,0.3,9.6,0.4,1.2,0.9,0.6
1,92.0,21.0,23.0,26.0,28.0,29.0,23.0,38.0,42.0,40.0,...,1.9,NaN,NaN,NaN,0.4,9.7,0.5,1.0,1.0,0.7
2,97.0,22.0,25.0,28.0,25.0,22.0,22.0,36.0,38.0,36.0,...,0.4,NaN,NaN,NaN,3.2,11.4,-2.5,80.0,45.0,100.0
3,99.0,21.0,22.0,23.0,22.0,21.0,21.0,33.0,34.0,33.0,...,0.3,NaN,NaN,NaN,-4.5,13.2,5.8,110.0,60.0,140.0
4,NaN,22.0,22.0,22.0,22.0,22.0,22.0,34.0,36.0,34.0,...,NaN,NaN,NaN,1.2,-8.1,4.3,-6.9,180.0,0.0,0.0


In [11]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, classification_report

In [12]:
imu = pd.read_csv("imu_sensor_data.csv").drop(columns=["label"])
proximity = pd.read_csv("proximity_sensor_data.csv").drop(columns=["label"])
em = pd.read_csv("em_sensor_data.csv").drop(columns=["label"])
gps = pd.read_csv("gps_environment_data.csv").drop(columns=["label"])
thermal = pd.read_csv("thermal_sensor_data.csv").drop(columns=["label"])
mic = pd.read_csv("microphone_audio_sensor.csv")

In [13]:
y = mic["label"]
X_mic = mic.drop(columns=["label"])

In [14]:
min_rows = min(len(imu), len(proximity), len(em), len(gps), len(thermal), len(X_mic))
dfs = [imu, proximity, em, gps, thermal, X_mic]
dfs = [df.reset_index(drop=True).iloc[:min_rows] for df in dfs]
y = y.reset_index(drop=True).iloc[:min_rows]


In [15]:
X = pd.concat(dfs, axis=1)

In [16]:
X_encoded = pd.get_dummies(X)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [17]:
model = RandomForestClassifier(random_state=42)
loo = LeaveOneOut()
y_true, y_pred = [], []

for train_idx, test_idx in loo.split(X_encoded):
    X_train, X_test = X_encoded.iloc[train_idx], X_encoded.iloc[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    y_true.append(y_test[0])
    y_pred.append(pred[0])


In [18]:
y_true_labels = label_encoder.inverse_transform(y_true)
y_pred_labels = label_encoder.inverse_transform(y_pred)

print("✅ Accuracy:", accuracy_score(y_true_labels, y_pred_labels))
print("\n📊 Classification Report:\n", classification_report(y_true_labels, y_pred_labels, zero_division=0))

✅ Accuracy: 0.0

📊 Classification Report:
                     precision    recall  f1-score   support

failsafe triggered       0.00      0.00      0.00       1.0
              safe       0.00      0.00      0.00       1.0
            threat       0.00      0.00      0.00       2.0

          accuracy                           0.00       4.0
         macro avg       0.00      0.00      0.00       4.0
      weighted avg       0.00      0.00      0.00       4.0

